### 1. Import Libraries

In [ ]:
import pandas as pd
import random
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

### 2. Load the data

In [2]:
df = pd.read_csv("expanded_qa_dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


### 3. Number of rows and columns

In [3]:
df.shape

(500, 2)

### 4. Null values

In [4]:
df.isnull().sum()

question    0
answer      0
dtype: int64

### 5. Duplicate values

In [5]:
df.duplicated().sum()

np.int64(0)

### 6. Tokenize the dataset

In [6]:
# Word-level tokenization
def tokenize(text: str) -> str:
    text = text.lower()
    text = text.replace("?", "")
    text = text.replace("'", "")
    return text.split()


word_level_tokens = tokenize("What is the capital of France")
print(word_level_tokens)
print(f"Total word-level tokens: {len(word_level_tokens)}")

['what', 'is', 'the', 'capital', 'of', 'france']
Total word-level tokens: 6


### 7. Build vocabulary (word-level, used to encode questions)

In [7]:
vocab = {
    '<UNK>': 0  # UNK -> Unknown vocabulary
}

def build_vocabulary(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

# Apply fnx on dataset
df.apply(build_vocabulary, axis=1)
print(f"Total vocabulary: {len(vocab)}")

Total vocabulary: 1224


### 8. Build answer vocabulary (class-level)

Instead of generating an answer word-by-word, the model predicts the **whole answer** as a single class. This is what lets it output complete multi-word answers (e.g. "Hypertext Markup Language") instead of just the first word.

In [8]:
answer_vocab = {}
for ans in df['answer']:
    if ans not in answer_vocab:
        answer_vocab[ans] = len(answer_vocab)

idx_to_answer = {index: answer for answer, index in answer_vocab.items()}
print(f"Total unique answers: {len(answer_vocab)}")

Total unique answers: 417


### 9. Convert words to numerical indices

In [9]:
def text_to_indices(text, vocab):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text

print(text_to_indices("What is the capital city of France", vocab))

[1, 2, 3, 4, 125, 5, 6]


### 10. Train / test split

We hold out 15% of the data to measure how well the model actually generalizes, instead of only looking at training loss.

Note: most answers in this dataset are unique to a single question (only ~33 out of 417 answers repeat), so many test questions will have answers the model never saw during training — those can never be predicted correctly. This is a property of the dataset, not a bug. We track this separately below.

In [10]:
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")

Train rows: 425, Test rows: 75


### 11. Dataset and DataLoader

In [11]:
class QADataset(Dataset):
    def __init__(self, df, vocab, answer_vocab):
        self.df = df
        self.vocab = vocab
        self.answer_vocab = answer_vocab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        answer_class = self.answer_vocab[self.df.iloc[index]['answer']]

        return torch.tensor(numerical_question), torch.tensor(answer_class)


train_dataset = QADataset(train_df, vocab, answer_vocab)
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

### 12. Build RNN Architecture (optimized)

Optimizations vs. the earlier version:
- **LSTM instead of plain RNN** — better at carrying information across longer questions, less prone to vanishing gradients.
- **Dropout** on the embeddings and on the final hidden state — reduces reliance on memorizing exact word patterns.
- **Larger embedding/hidden size** — more capacity to separate ~400+ answer classes.

In [12]:
class RNN(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim=64, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        self.fc_layer = nn.Linear(hidden_dim, output_size)

    def forward(self, question):
        embedded_question = self.dropout1(self.embedding(question))
        output, (hidden, cell) = self.lstm(embedded_question)
        hidden = self.dropout2(hidden.squeeze(0))
        return self.fc_layer(hidden)

### 13. Model

In [13]:
model = RNN(vocab_size=len(vocab), output_size=len(answer_vocab))

### 14. Training loop

Optimizations:
- `weight_decay` on Adam adds L2 regularization (penalizes large weights, reduces overfitting).
- Loss target is now a single answer-class index (no more first-token-only workaround).

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

epochs = 30

for epoch in range(epochs):
    training_loss = 0.0

    for question, answer in train_dataloader:
        optimizer.zero_grad()
        output = model(question)
        loss = criterion(output, answer)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    print(f"Epoch {epoch + 1} / {epochs}: Loss:{training_loss:.2f}")

Epoch 1 / 30: Loss:2570.39


Epoch 2 / 30: Loss:2378.90


Epoch 3 / 30: Loss:2203.67


Epoch 4 / 30: Loss:1952.16


Epoch 5 / 30: Loss:1641.03


...

Epoch 26 / 30: Loss:104.80


Epoch 27 / 30: Loss:75.54


Epoch 28 / 30: Loss:74.32


Epoch 29 / 30: Loss:64.45


Epoch 30 / 30: Loss:63.08


### 15. Evaluate model accuracy

Three numbers, because a single "accuracy" would be misleading here:
- **Train accuracy** — how well the model learned the training questions.
- **Test accuracy (overall)** — includes test questions whose answer was *never* seen in training (impossible to get right).
- **Test accuracy (seen answers only)** — restricted to test questions whose answer class also appeared during training. This is the fairer measure of real generalization.

In [15]:
def evaluate_accuracy(model, eval_df, vocab, answer_vocab, idx_to_answer):
    model.eval()
    correct = 0
    total = len(eval_df)

    with torch.no_grad():
        for i in range(total):
            question = eval_df.iloc[i]['question']
            true_answer = eval_df.iloc[i]['answer']

            numerical_question = text_to_indices(question, vocab)
            question_tensor = torch.tensor(numerical_question).unsqueeze(0)

            output = model(question_tensor)
            predicted_index = torch.argmax(output, dim=1).item()
            predicted_answer = idx_to_answer.get(predicted_index)

            if predicted_answer == true_answer:
                correct += 1

    model.train()
    return (correct / total) * 100 if total > 0 else 0.0


train_accuracy = evaluate_accuracy(model, train_df, vocab, answer_vocab, idx_to_answer)

test_accuracy_overall = evaluate_accuracy(model, test_df, vocab, answer_vocab, idx_to_answer)

seen_answers = set(train_df['answer'])
test_df_seen = test_df[test_df['answer'].isin(seen_answers)]
test_accuracy_seen = evaluate_accuracy(model, test_df_seen, vocab, answer_vocab, idx_to_answer)

print(f"Train accuracy: {train_accuracy:.2f}%")
print(f"Test accuracy (overall, {len(test_df)} rows): {test_accuracy_overall:.2f}%")
print(f"Test accuracy (only answers seen during training, {len(test_df_seen)} rows): {test_accuracy_seen:.2f}%")

Train accuracy: 99.53%
Test accuracy (overall, 75 rows): 2.67%
Test accuracy (only answers seen during training, 21 rows): 9.52%


### 16. Retrain on the full dataset (for deployment)

The train/test split above was useful to *measure* generalization honestly. But for deployment, the goal is different: we want the model to correctly answer every question that exists in our dataset — including the ones that happened to land in the 15% test split (like "What is the capital of France?", which the train-only model above never saw).

So for the final, deployed model we retrain from scratch on the **full dataset**, using the same architecture and hyperparameters. This is why the accuracy numbers above (on the train/test split) are lower than what the deployed app actually achieves.

In [16]:
full_dataset = QADataset(df, vocab, answer_vocab)
full_dataloader = DataLoader(full_dataset, batch_size=1, shuffle=True)

model = RNN(vocab_size=len(vocab), output_size=len(answer_vocab))
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

epochs = 30
for epoch in range(epochs):
    training_loss = 0.0

    for question, answer in full_dataloader:
        optimizer.zero_grad()
        output = model(question)
        loss = criterion(output, answer)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1} / {epochs}: Loss:{training_loss:.2f}")

full_accuracy = evaluate_accuracy(model, df, vocab, answer_vocab, idx_to_answer)
print(f"Accuracy on full (known) dataset: {full_accuracy:.2f}%")

Epoch 10 / 30: Loss:704.39


Epoch 20 / 30: Loss:197.63


Epoch 30 / 30: Loss:99.54


Accuracy on full (known) dataset: 98.80%


### 17. Predict function (with fuzzy fallback)

Same idea as the deployed FastAPI app: if the model's confidence is below the threshold, fall back to finding the most similar question in the dataset (via `difflib`) instead of just giving up. This handles small wording differences (e.g. "capital **city** of France").

In [17]:
import difflib

known_questions = df['question'].tolist()

def find_closest_question(question, cutoff=0.6):
    matches = difflib.get_close_matches(question, known_questions, n=1, cutoff=cutoff)
    if not matches:
        return None, None
    matched_question = matches[0]
    matched_answer = df.loc[df['question'] == matched_question, 'answer'].iloc[0]
    return matched_question, matched_answer


def predict(model, question, vocab, idx_to_answer, threshold=0.3):
    numerical_question = text_to_indices(question, vocab)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        output = model(question_tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        confidence, predicted_index = torch.max(probs, dim=1)
    model.train()

    if confidence.item() >= threshold:
        return idx_to_answer[predicted_index.item()], confidence.item()

    _, matched_answer = find_closest_question(question)
    if matched_answer is not None:
        return matched_answer, confidence.item()

    return "I don't know", confidence.item()


for sample_question in [
    "What is the capital of France?",
    "What is the capital city of France",
    "What does HTML stands for?",
    "What does CPU stand for?",
]:
    answer, confidence = predict(model, sample_question, vocab, idx_to_answer)
    print(f"Q: {sample_question}\nA: {answer} (confidence: {confidence:.2f})\n")

Q: What is the capital of France?
A: Paris (confidence: 1.00)

Q: What is the capital city of France
A: Paris (confidence: 0.99)

Q: What does HTML stands for?
A: HyperText Markup Language (confidence: 0.93)

Q: What does CPU stand for?
A: Central Processing Unit (confidence: 0.97)



### 18. Save the model

We save four things now (the dataset is needed too, for the fuzzy fallback in the predict function):
- `qa_model.pth` — the trained model weights (trained on the full dataset).
- `vocab.pkl` — the word vocabulary.
- `answer_vocab.pkl` — the answer class mapping.
- `qa_dataset.csv` — a copy of the dataset, used by the fuzzy fallback to match reworded questions.

In [18]:
torch.save(model.state_dict(), "qa_model.pth")

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open("answer_vocab.pkl", "wb") as f:
    pickle.dump(answer_vocab, f)

df.to_csv("qa_dataset.csv", index=False)

print("Model and supporting files saved:")
print(" - qa_model.pth")
print(" - vocab.pkl")
print(" - answer_vocab.pkl")
print(" - qa_dataset.csv")

Model and supporting files saved:
 - qa_model.pth
 - vocab.pkl
 - answer_vocab.pkl
 - qa_dataset.csv
